In [ ]:
import pandas as pd

In [ ]:
df_desligados = pd.read_parquet(r'.\df_desligados.parquet')

In [ ]:
df_ronda = pd.read_parquet(r'.\df_ronda.parquet')

In [ ]:
df_desligados.head(1)

In [ ]:
df_ronda.head(1)

In [ ]:
# Se ainda não estiver carregado do parquet:
# df_ronda = pd.read_parquet("df_ronda.parquet")
 
# 1) Garantir tipos consistentes nas chaves
df_ronda["cadastro"] = df_ronda["cadastro"].astype(str).str.strip()
df_desligados["Cadastro"] = df_desligados["Cadastro"].astype(str).str.strip()
 
# 2) Preparar base de desligados com a chave correta
df_desligados_aux = (
    df_desligados.loc[:, ["Cadastro", "Data_Dem"]]
    .assign(Data_Dem=lambda d: pd.to_datetime(d["Data_Dem"], errors="coerce"))
    .dropna(subset=["Cadastro"])
    .sort_values("Data_Dem")
    .drop_duplicates(subset=["Cadastro"], keep="last")  # mantém a data mais recente por cadastro
)
 
# 3) Se df_ronda já tiver Data_Dem, remove antes para evitar _x / _y
if "Data_Dem" in df_ronda.columns:
    df_ronda = df_ronda.drop(columns=["Data_Dem"])
 
# 4) Fazer o join
df_ronda = df_ronda.merge(
    df_desligados_aux,
    left_on="cadastro",
    right_on="Cadastro",
    how="left",
    validate="m:1"
)
 
# 5) Remover a coluna auxiliar vinda do df_desligados
df_ronda = df_ronda.drop(columns=["Cadastro"])
 
# 6) Conferir
print(df_ronda[["cadastro", "Data_Dem"]].head())
print("Qtd com Data_Dem preenchida:", df_ronda["Data_Dem"].notna().sum())
 

In [ ]:
# 7) Salvar de volta no parquet
df_ronda.to_parquet("df_ronda.parquet", index=False)